# Chapter 7: Using Examples (Few-Shot Prompting)

## Lesson

**Few-shot prompting** means providing examples of desired input/output pairs in your prompt. This is one of the most effective ways to control Claude's output.

### Terminology
- **Zero-shot**: No examples provided
- **One-shot**: One example provided
- **Few-shot**: Multiple examples provided

### When to Use Examples
- When you need a very specific output format
- When describing the desired tone/style is hard but showing it is easy
- When Claude needs to learn a pattern to extrapolate
- Examples are often more effective than lengthy descriptions

In [ ]:
import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt: str = "", prefill: str = ""):
    messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 2000,
        "temperature": 0.0,
        "messages": messages
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    message = client.messages.create(**kwargs)
    return message.content[0].text

### Example: Tone Matching with Examples

Instead of describing the tone you want, show it:

In [ ]:
# Without example — generic response
print("--- Without example ---")
response = get_completion("How does Santa Claus deliver presents around the world in one night?")
print(response)
print()

# With example — Claude matches the warm, magical tone
print("--- With example (warm, parent-friendly tone) ---")
response = get_completion("""Here's an example of how to answer children's questions about magical figures:

<example>
Q: How does the Tooth Fairy know when I've lost a tooth?
A: The Tooth Fairy has a special kind of magic that lets her feel a little tingle whenever a child loses a tooth, no matter where they are! She has a magical map that shows every child in the world, and when a tooth wiggles loose, a tiny star appears on the map. That's how she always knows to visit!
</example>

Now answer this question in the same style:
Q: How does Santa Claus deliver presents around the world in one night?""")
print(response)

### Example: Format Extrapolation

Claude can learn structured output formats from examples and apply them to new inputs:

In [ ]:
response = get_completion("""Extract names and professions from the text.

<examples>
Input: In the bustling town of Millbrook, baker Thomas spent his mornings crafting sourdough while teacher Maria prepared her lesson plans at the school next door.
Output:
<person>
  <name>Thomas</name>
  <profession>baker</profession>
</person>
<person>
  <name>Maria</name>
  <profession>teacher</profession>
</person>
</examples>

Now extract from this text:
Dr. Sarah Chen reviewed the patient charts while nurse practitioner James Rodriguez prepared the morning medications. Down the hall, surgeon Dr. Park was already scrubbing in for the first operation.""")

print(response)

---
## Exercises

### Exercise 7.1
Use few-shot examples to classify emails, where the **last character** of the response is the category letter (A-D). This exercise combines few-shot prompting with the classification task from Chapter 6.

In [ ]:
# Exercise 7.1
EMAILS = [
    "Hi — I was charged twice for my order last week. Can you help me get a refund for the duplicate?",
    "I'm interested in your premium plan. Does it include API access?",
    "The widget I received is cracked and doesn't turn on. I need a replacement.",
    "I love your product! Just wanted to say thanks."
]
EXPECTED = ["C", "A", "B", "D"]

# TODO: Write a prompt with few-shot examples where the last character is the category letter
for i, email in enumerate(EMAILS):
    PROMPT = f"""Classify emails into categories. The last character of your response must be the category letter.

Categories:
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other

<examples>
Email: "Do you offer volume discounts for teams over 50 people?"
This is a question about purchasing before buying. Category: A

Email: "My account shows a charge I don't recognize from last Tuesday."
This is about a billing concern. Category: C
</examples>

Email: "{email}"
"""
    
    response = get_completion(PROMPT)
    last_char = response.strip()[-1]
    passed = last_char == EXPECTED[i]
    print(f"Email {i+1}: Last char='{last_char}' | Expected='{EXPECTED[i]}' | {'✅' if passed else '❌'}")

---
### Example Playground

In [ ]:
# Playground - try few-shot prompting for any task!
PROMPT = """Convert sentences to emoji stories.

<examples>
Input: I went to the store and bought apples.
Output: 🚶➡️🏪💰🍎

Input: The cat sat on the mat and slept.
Output: 🐱⬇️🧶😴
</examples>

Input: I flew to Paris and ate croissants at a cafe.
"""

response = get_completion(PROMPT)
print(response)